# Churn + CLV Prediction: Data Cleaning

**Data:** Online Retail II (UCI), a real UK online retailer, Dec 2009 - Dec 2011.

## 1. Load and combine the raw data

In [2]:
# Location of the ORIGINAL UCI Excel file
path = "../data/raw/online_retail_II.xlsx"

# Check the exact sheet names before loading
xls = pd.ExcelFile(path)
print(xls.sheet_names)

# sheet_name=None loads ALL sheets into a dictionary: {sheet name: table}
raw_by_year = pd.read_excel(xls, sheet_name=None)

# Print each sheet's name and its size (rows, columns)
for sheet_name, sheet_table in raw_by_year.items():
    print(sheet_name, sheet_table.shape)

['Year 2009-2010', 'Year 2010-2011']
Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)


In [3]:
# Remember which sheet each row came from, then stack them into one table
tables = []
for sheet_name, sheet_df in raw_by_year.items():
    temp = sheet_df.copy()                 # copy, so the raw data stays untouched
    temp["source_sheet"] = sheet_name      # new column: name of the sheet this row came from
    tables.append(temp)

transactions = pd.concat(tables, ignore_index=True)

# Clean column names: lowercase, spaces become underscores
# Example: "Customer ID" becomes "customer_id"
transactions.columns = [c.strip().lower().replace(" ", "_") for c in transactions.columns]

# Total number of rows
len(transactions)

1067371

In [4]:
transactions.head()

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


## 2. Remove the overlap between the two sheets

In [5]:
# Overlap window: 1 Dec 2010 up to the end of 9 Dec 2010
# (the end date is exclusive, so all of 9 Dec is included)
in_overlap = (transactions["invoicedate"] >= "2010-12-01") & (transactions["invoicedate"] < "2010-12-10")

# Count how many overlap rows each sheet has
transactions[in_overlap].groupby("source_sheet").size()

source_sheet
Year 2009-2010    22523
Year 2010-2011    22523
dtype: int64

In [6]:
# Keep all rows from the newer sheet.
# From the older sheet keep only rows BEFORE 1 Dec 2010
# (the rest is repeated in the newer sheet).
keep_rows = (transactions["source_sheet"] == "Year 2010-2011") | (transactions["invoicedate"] < "2010-12-01")
transactions = transactions[keep_rows].reset_index(drop=True)

len(transactions)

1044848

## 3. Remove rows without a customer ID

In [7]:
# Count the empty values in each column
transactions.isnull().sum()

invoice              0
stockcode            0
description       4275
quantity             0
invoicedate          0
price                0
customer_id     235287
country              0
source_sheet         0
dtype: int64

In [8]:
# Remove rows where customer_id is empty
transactions = transactions.dropna(subset=["customer_id"]).copy()

# customer_id looks like 13085.0 because of the empty values.
# Now no empty values are left, so make it a whole number.
transactions["customer_id"] = transactions["customer_id"].astype(int)

# Quick look at the size after this step
print("Rows now:", f"{len(transactions):,}")

Rows now: 809,561


In [9]:
# Check that no values are missing any more
print("Rows:", f"{len(transactions):,}")
print(transactions.isnull().sum())

Rows: 809,561
invoice         0
stockcode       0
description     0
quantity        0
invoicedate     0
price           0
customer_id     0
country         0
source_sheet    0
dtype: int64


## 4. Remove duplicate rows

In [10]:
# Count rows that are exact copies of an earlier row
transactions.duplicated().sum()

np.int64(11676)

In [11]:
# Keep the first copy of each repeated row and drop the extra copies
transactions = transactions.drop_duplicates().reset_index(drop=True)

In [12]:
# Final check after removing duplicates
print("Rows:", f"{len(transactions):,}")
print("Duplicates:", transactions.duplicated().sum())
print("Unique customers:", transactions["customer_id"].nunique())
print("Date range:", transactions["invoicedate"].min(), "->", transactions["invoicedate"].max())

Rows: 797,885
Duplicates: 0
Unique customers: 5942
Date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


## 5. Separate sales and returns

In [13]:
# Check cancelled invoices (invoice number starts with "C")
print("Cancelled invoices: ", transactions["invoice"].astype(str).str.startswith("C").sum())

# Check rows with negative quantity
print("Rows with negative quantity: ", (transactions["quantity"] < 0).sum())

# Check rows with price zero or below
print("Rows with zero price: ", (transactions["price"] <= 0).sum())

Cancelled invoices:  18390
Rows with negative quantity:  18390
Rows with zero price:  70


In [14]:
# Check rows that are BOTH a cancelled invoice AND have negative quantity
# (if this is also 18390, the two groups are exactly the same rows)
(transactions["invoice"].astype(str).str.startswith("C") & (transactions["quantity"] < 0)).sum()

np.int64(18390)

In [15]:
# Returns: all cancelled invoices (these are exactly the negative-quantity rows)
is_cancelled = transactions["invoice"].astype(str).str.startswith("C")
returns = transactions[is_cancelled].copy()

# Sales: not cancelled AND price above zero (free items are not real purchases)
sales = transactions[~is_cancelled & (transactions["price"] > 0)].copy()

In [16]:
# Sizes of the two new tables, and what was left out
print("Returns:", len(returns))
print("Sales:", len(sales))
print("Zero-price rows left out:", len(transactions) - len(returns) - len(sales))

Returns: 18390
Sales: 779425
Zero-price rows left out: 70


## 6. Save the cleaned data

In [17]:
# invoice and stockcode contain both numbers and text (for example "C489449" and "79323P").
# Parquet cannot store mixed types in one column, so convert both columns to text first.
for table in (sales, returns):
    table["invoice"] = table["invoice"].astype(str)
    table["stockcode"] = table["stockcode"].astype(str)

# Save both tables so we do not have to reload the slow Excel file again
out_dir = "../data/processed/"
sales.to_parquet(out_dir + "sales.parquet", index=False)
returns.to_parquet(out_dir + "returns.parquet", index=False)

print("Saved sales:", f"{len(sales):,}", "rows")
print("Saved returns:", f"{len(returns):,}", "rows")

Saved sales: 779,425 rows
Saved returns: 18,390 rows


## Cleaning decisions

| Step | Decision | Rows |
|---|---|---|
| Start | Both Excel sheets combined | 1,067,371 |
| Overlap week | 1-9 Dec 2010 was in both sheets, so it was kept once | 22,523 removed |
| Missing customer ID | Rows without a customer cannot be used for churn | 235,287 removed (22.5%) |
| Duplicate rows | Exact repeated rows dropped, first copy kept | 11,676 removed |
| Cancelled invoices | Moved to a separate `returns` table (all have negative quantity) | 18,390 |
| Zero-price rows | Free items, not real purchases, left out | 70 |
| **Final tables** | `sales`: real purchases, `returns`: cancellations | **779,425** and **18,390** |